In [ ]:
%%file constant.py

from acadia.runtime import Runtime

class ConstantRuntime(Runtime):
    
    @staticmethod
    def main():
        from acadia.system import Acadia
        from acadia.arrays import ConstantWaveform
        
        acadia = Acadia()
        pulse_channel = acadia.DAC(1)
        pulse = ConstantWaveform(pulse_channel, 1e-6)
        
        # Create a sequence for the sequencer
        def sequence(a):
            with a.channel_synchronizer():
                a.generate(pulse_channel, pulse)
                
            with a.repeat():
                with a.sequencer().test(a.dma_fifo_occupancy(pulse_channel) == 0):
                    with a.channel_synchronizer(block=False):
                        a.generate(pulse_channel, pulse)

        # Attach to the hardware
        acadia.attach()
        pulse.populate(0.99)
        pulse_channel.set_nyquist_zone(2)
        pulse_channel.configure_nco(frequency=2000e6)
        pulse_channel.set_vop(20000)

        acadia.compile(sequence)
        acadia.run(block=False)
        

In [ ]:
from .constant import ConstantRuntime
rt = ConstantRuntime("192.168.2.69", "constant.py")
rt.run()